# Reconhecimento de Gestos da Mão em Tempo Real (Webcam)

Este notebook foi alterado para rastrear **até 2 mãos** simultaneamente utilizando o MediaPipe.
Como o modelo oficial possui um limite de fábrica de apenas 7 gestos registrados, eu adicionei uma 
**Lógica de Matemática de Pixels personalizada (Heurística)**! O código calcula exatamente onde estão as 
pontas dos seus dedos na tela e descobre novos gestos bônus inéditos: como o Rock and Roll 🤘, 
Mostrar o Dedo do Meio 🤬, etc!

In [13]:
!pip install opencv-python mediapipe

In [14]:
import os
import urllib.request

model_url = "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
model_path = "gesture_recognizer.task"

if not os.path.exists(model_path):
    print("Baixando o modelo Gesture Recognizer...")
    urllib.request.urlretrieve(model_url, model_path)
    print("Modelo baixado com sucesso!")
else:
    print("O modelo de Gestos já se encontra baixado na pasta.")

O modelo de Gestos já se encontra baixado na pasta.


In [15]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# FUNÇÃO BÔNUS: Ler gestos extras analisando e calculando pontos da mão
def checar_gestos_extras(landmarks):
    dedos_levantados = []
    # Landmark 8=Ponta do Indicador, 6=Articulação Inferior
    # Se o Y da Ponta for Menor (mais alto na tela) que o da Articulação, o dedo está pra cima
    if landmarks[8].y < landmarks[6].y: dedos_levantados.append("Indicador")
    if landmarks[12].y < landmarks[10].y: dedos_levantados.append("Medio")
    if landmarks[16].y < landmarks[14].y: dedos_levantados.append("Anelar")
    if landmarks[20].y < landmarks[18].y: dedos_levantados.append("Mindinho")
    
    # Nossas regras de novos gestos:
    if dedos_levantados == ["Medio"]:
        return "Feio (Dedo do Meio)"
    elif dedos_levantados == ["Indicador", "Mindinho"]:
        return "Rock / Homem Aranha"
    elif dedos_levantados == ["Indicador", "Medio", "Anelar"]:
        return "Tres Dedos Lidos"
    
    return None


# 1. Configurações para rastrear Múltiplas Mãos!!!
base_options = python.BaseOptions(model_asset_path=model_path)
# MUDANÇA: 'num_hands=2' permite reconhecer as duas mãos ao mesmo tempo
options = vision.GestureRecognizerOptions(base_options=base_options, num_hands=2)
recognizer = vision.GestureRecognizer.create_from_options(options)

# Conectar na câmera de vídeo
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro ao acessar a câmera!")
else:
    print("Câmera de Múltiplos Gestos ativada!")

    while True:
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1) # Funcionar como espelho natural

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # 2. Roda a Inteligência do Google via CPU
        recognition_result = recognizer.recognize(mp_image)

        # 3. Ler ambas as mãos encontradas!
        if recognition_result.gestures:
            # Laço para lidar com a Mão 1 e Mão 2 caso existam
            for i, hand_gesture in enumerate(recognition_result.gestures):
                gesture = hand_gesture[0]
                gesture_name = gesture.category_name
                score = gesture.score
                
                # Procurar Landmarks exatas DESTA mão
                landmarks = recognition_result.hand_landmarks[i]
                
                # Descobrir Centro X e Y da mão aproximado para fixar o texto nela!
                centro_x = int(landmarks[9].x * frame.shape[1])
                centro_y = int(landmarks[9].y * frame.shape[0])
                
                # Aplicar minha inteligência de Heurística Personalizada de Dedos por Mão
                gesto_customizado = checar_gestos_extras(landmarks)
                
                # Sobrepor com o gesto personalizado caso a inteligência tenha encontrado um
                if gesto_customizado:
                    gesture_name = gesto_customizado
                
                # Adicionar texto em cima DESTA mão (Apoia Duas as mesmo tempo em cantos diferentes)
                if gesture_name != "None":
                    cv2.putText(frame, f"Mao {i+1}: {gesture_name}",
                                (centro_x - 100, centro_y - 80), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 3, cv2.LINE_AA)

                # Desenhar e Pintar os 21 pontos Mágicos Visuais nos dedos da mão ativa
                for landmark in landmarks:
                    x = int(landmark.x * frame.shape[1])
                    y = int(landmark.y * frame.shape[0])
                    cv2.circle(frame, (x, y), 5, (200, 50, 255), -1) # Cor Rosa/Roxo

        cv2.imshow('Deteccao MediaPipe Duas Maos e Novos Gestos', frame)

        # Fechamento Dinamico
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


Câmera de Múltiplos Gestos ativada!
